Sample data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
pip install gdown

In [ ]:
!wget https://raw.githubusercontent.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT/main/data/gold_with_sentiments/gold_test.csv
!wget https://raw.githubusercontent.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT/main/data/gold_with_sentiments/gold_train.csv

--2026-05-18 13:43:32--  https://raw.githubusercontent.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT/main/data/gold_with_sentiments/gold_test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 125119 (122K) [text/plain]
Saving to: ‘gold_test.csv’

gold_test.csv       100%[===================>] 122.19K  --.-KB/s    in 0.02s   

2026-05-18 13:43:32 (4.84 MB/s) - ‘gold_test.csv’ saved [125119/125119]

--2026-05-18 13:43:32--  https://raw.githubusercontent.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT/main/data/gold_with_sentiments/gold_train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... 

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import gdown
import tqdm
import ast
from pyspark.sql.functions import udf
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType,StructField,StringType,ArrayType,FloatType
spark = SparkSession.builder.getOrCreate()

In [ ]:
gdown.download(id = "1eeKvI9H7nUaVaIKdDM1yGTGbDy2sHdB1", output = "/content/high_confidence_samples_kindle_store.parquet")
gdown.download(id = "1pWOu6UVxAB7vo3CTTx5hQXIUob8j5QGz", output = "/content/high_confidence_samples_software.parquet")
gdown.download(id = "1etn4E1vW7Ab_jKVi1nTJpSjooEamCPbP", output = "/content/high_confidence_samples_electronics_p1.parquet")
gdown.download(id = "1Z-cRLF21uaUBgoRpE5nYPVns-XHqgX0k", output = "/content/high_confidence_samples_electronics_p2.parquet")
gdown.download(id = "1yEdq7SGfP-62h4dvKXOF63Q4DzP1LuBf", output = "/content/high_confidence_samples_office_products.parquet")

Downloading...
From (original): https://drive.google.com/uc?id=1eeKvI9H7nUaVaIKdDM1yGTGbDy2sHdB1
From (redirected): https://drive.google.com/uc?id=1eeKvI9H7nUaVaIKdDM1yGTGbDy2sHdB1&confirm=t&uuid=1c918c67-19c7-4950-a760-76d5d0142391
To: /content/high_confidence_samples_kindle_store.parquet
100%|██████████| 1.83G/1.83G [00:24<00:00, 75.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1eBf9L8koOjPNQ8EwwcIaz32Zinu9jFvu
From (redirected): https://drive.google.com/uc?id=1eBf9L8koOjPNQ8EwwcIaz32Zinu9jFvu&confirm=t&uuid=914a597a-a368-41a6-84b1-18e76e542214
To: /content/high_confidence_samples_software.parquet
100%|██████████| 164M/164M [00:03<00:00, 52.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1etn4E1vW7Ab_jKVi1nTJpSjooEamCPbP
From (redirected): https://drive.google.com/uc?id=1etn4E1vW7Ab_jKVi1nTJpSjooEamCPbP&confirm=t&uuid=040be415-abac-4837-9b61-c02093032a32
To: /content/high_confidence_samples_electronics_p1.parquet
100%|██████████| 1.21G/1.

'/content/high_confidence_samples_office_products.parquet'

In [ ]:
outdir = "/content/drive/MyDrive/data_train/asc"

In [ ]:
def make_pair_key(sentence_text, aspect):
    return F.concat_ws("||", sentence_text, aspect)

In [ ]:
def sentiment_label(triplet):
    return int(np.argmax(triplet))
sentiment_udf = udf(sentiment_label)

In [ ]:
def expand_asc_rows(df):
    df = df.withColumn("zipped", F.arrays_zip("aspects", "sentiments"))
    df = df.withColumn("z", F.explode("zipped"))
    df = df.withColumn("aspects", F.col("z.aspects")) \
           .withColumn("sentiments", F.col("z.sentiments"))
    df = df.withColumn("label",sentiment_udf(F.col("sentiments")))
    return df.select(
        "parent_asin",
        "sentence_id",
        F.col("sentence_text"),
        "rating",
        "category_name",
        "aspects",
        "sentiments",
        "label"
    )

In [ ]:
def sample_category_400k(df, target_total=400_000, seed=42):
    """
    Rule:
    - lấy toàn bộ neutral
    - pos/neg sampling 1:1 từ phần còn lại
    """
    print("spliting")
    neutral_df = df.filter(F.col("label") == 1)
    pos_df = df.filter(F.col("label") == 2)
    neg_df = df.filter(F.col("label") == 0)
    print("Complete split")
    neutral_count = neutral_df.count()
    remaining = target_total - neutral_count
    polar_target_each = remaining // 2

    pos_count = pos_df.count()
    neg_count = neg_df.count()
    print("Complete counting")
    pos_fraction = min(polar_target_each / pos_count, 1.0) if pos_count > 0 else 0
    neg_fraction = min(polar_target_each / neg_count, 1.0) if neg_count > 0 else 0
    print("Sampling")
    sampled_pos = pos_df.sample(withReplacement=False, fraction=pos_fraction, seed=seed)
    sampled_neg = neg_df.sample(withReplacement=False, fraction=neg_fraction, seed=seed)

    sampled_pos = sampled_pos.limit(polar_target_each)
    sampled_neg = sampled_neg.limit(polar_target_each)

    sampled = neutral_df.unionByName(sampled_pos).unionByName(sampled_neg)
    sampled = sampled.cache()
    print("return stats")
    stats = {
        "neutral": sampled.filter(F.col("label") == 1).count(),
        "pos": sampled.filter(F.col("label") == 2).count(),
        "neg": sampled.filter(F.col("label") == 0).count(),
        "total": sampled.count(),
    }
    return sampled, stats

In [ ]:
gold_train = pd.read_csv("gold_train.csv")
gold_test = pd.read_csv("gold_test.csv")

In [ ]:
gold_train['aspects'] = gold_train['aspects'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)
gold_train['sentiments'] = gold_train['sentiments'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [ ]:
gold_test['aspects'] = gold_test['aspects'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)
gold_test['sentiments'] = gold_test['sentiments'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [ ]:
gold_train = spark.createDataFrame(gold_train)
gold_test = spark.createDataFrame(gold_test)

In [ ]:
gold_train_expanded = expand_asc_rows(gold_train)
gold_test_expanded = expand_asc_rows(gold_test)

In [ ]:
gold_train_expanded = gold_train_expanded.withColumn(
    "pair_key",
    make_pair_key(
        F.col("sentence_text"),
        F.col("aspects")
    )
)

gold_test_expanded = gold_test_expanded.withColumn(
    "pair_key",
    make_pair_key(
        F.col("sentence_text"),
        F.col("aspects")
    )
)

In [ ]:
categories = [
    "kindle_store", "software", "electronics_p1", "electronics_p2", "office_products"
]

sampled_category_dfs = []
category_stats = []

for category in categories:
    print(f"\n========== {category} ==========")
    in_path = f"/content/high_confidence_samples_{category}.parquet"

    df = spark.read.parquet(in_path)
    expanded_df = expand_asc_rows(df)
    expanded_df = expanded_df.withColumn(
        "pair_key",
        make_pair_key(
            F.col("sentence_text"),
            F.col("aspects")
        )
    )
    sampled_df, stats = sample_category_400k(
        expanded_df,
        target_total=400000,
        seed=191
    )
    print(stats)

    sampled_category_dfs.append(sampled_df)
    category_stats.append({
        "category": category,
        **stats,
    })

pseudo_train_df = sampled_category_dfs[0]
for df in sampled_category_dfs[1:]:
    pseudo_train_df = pseudo_train_df.unionByName(df)
pseudo_train_df = pseudo_train_df.dropDuplicates(["pair_key"])
print(f"\nPseudo train total after concat/dedup: {pseudo_train_df.count()}")

# leakage removal with gold test
print("Removing leakage")
before = pseudo_train_df.count()
pseudo_train_df = pseudo_train_df.join(
    gold_test_expanded.select("pair_key").distinct(),
    on="pair_key",
    how="left_anti"
)
after = pseudo_train_df.count()
print(f"Removed leakage: {before - after}")

# Combine with gold train
train_df = pseudo_train_df.unionByName(gold_train_expanded)
train_df = train_df.dropDuplicates(["pair_key"])

print(f"Final train size: {train_df.count()}")


========== kindle_store ==========
spliting
Complete split
Complete counting
Sampling
return stats
{'neutral': 19656, 'pos': 189270, 'neg': 189580, 'total': 398506}

========== software ==========
spliting
Complete split
Complete counting
Sampling
return stats
{'neutral': 0, 'pos': 200000, 'neg': 200000, 'total': 400000}

========== electronics_p1 ==========
spliting
Complete split
Complete counting
Sampling
return stats
{'neutral': 21817, 'pos': 188987, 'neg': 188621, 'total': 399425}

========== electronics_p2 ==========
spliting
Complete split
Complete counting
Sampling
return stats
{'neutral': 18316, 'pos': 190842, 'neg': 190513, 'total': 399671}

========== office_products ==========
spliting
Complete split
Complete counting
Sampling
return stats
{'neutral': 6956, 'pos': 196430, 'neg': 196522, 'total': 399908}


TypeError: object of type 'DataFrame' has no len()

In [ ]:
print(f"\nPseudo train total after concat/dedup: {pseudo_train_df.count()}")

# leakage removal with gold test
print("Removing leakage")
before = pseudo_train_df.count()
pseudo_train_df = pseudo_train_df.join(
    gold_test_expanded.select("pair_key").distinct(),
    on="pair_key",
    how="left_anti"
)
after = pseudo_train_df.count()
print(f"Removed leakage: {before - after}")

# Combine with gold train
train_df = pseudo_train_df.unionByName(gold_train_expanded)
train_df = train_df.dropDuplicates(["pair_key"])
train_df = train_df.drop("pair_key")

print(f"Final train size: {train_df.count()}")


Pseudo train total after concat/dedup: 1910098
Removing leakage
Removed leakage: 0
Final train size: 1912508


In [ ]:
train_df.coalesce(1).write.mode("overwrite").parquet(f'{outdir}/asc_data/asc_train_data.parquet')
gold_test_expanded.coalesce(1).write.mode("overwrite").parquet(f'{outdir}/asc_data/asc_test_data.parquet')

Train

In [ ]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
)
from datasets import Dataset,load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

In [ ]:
model_name = "roberta-base"
train_path = "/content/drive/MyDrive/data_train/asc/asc_data/asc_train_data2.parquet"
test_path = "/content/drive/MyDrive/data_train/asc/asc_data/asc_test_data2.parquet"
save_path = "/content/drive/MyDrive/data_train/asc"
model_save_path = f"{save_path}/asc_model"

In [ ]:
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

In [ ]:
def build_input_text(sentence, aspect):
    sentence = sentence
    aspect = aspect
    if not sentence:
        return ""
    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    if aspect and pattern.search(sentence):
        return pattern.sub(f"[ASP] {aspect} [/ASP]", sentence, count=1)
    return f"{sentence} </s></s> {aspect}"

def build_input_df(df):
    df = df.copy()
    df["input_text"] = df.apply(
        lambda r: build_input_text(r["sentence_text"], r["aspects"]),
        axis=1
    )
    return df

train_df = build_input_df(train_df)
test_df = build_input_df(test_df)

In [ ]:
train_ds = Dataset.from_pandas(train_split_df[["input_text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["input_text", "label"]].reset_index(drop=True))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
special_tokens = {"additional_special_tokens": ["[ASP]", "[/ASP]"]}
num_added = tokenizer.add_special_tokens(special_tokens)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
)

if num_added > 0:
    model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=192,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

Map:   0%|          | 0/1874257 [00:00<?, ? examples/s]

Map:   0%|          | 0/38251 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

In [ ]:
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=model_save_path,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=5000,
    save_steps=5000,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    num_train_epochs=3,
    # gradient_accumulation_steps=1,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    optim="adamw_torch_fused",
    # torch_compile=True,
    fp16=True,
    bf16=False,
    logging_steps=200,
    dataloader_prefetch_factor=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=191,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/data_train/asc/asc_model/checkpoint-20000")

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Step,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
25000,0.012250,0.881760,0.864450,0.818644,0.621707,0.637164,0.860639,0.864450,0.852766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=29884, training_loss=0.003908000795688994, metrics={'train_runtime': 9281.3984, 'train_samples_per_second': 412.116, 'train_steps_per_second': 3.22, 'total_flos': 3.774048872341064e+17, 'train_loss': 0.003908000795688994, 'epoch': 2.0})

In [ ]:
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/data_train/asc/asc_model/tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/data_train/asc/asc_model/tokenizer/tokenizer.json')

In [ ]:
test_pred = trainer.predict(test_ds)
test_metrics = compute_metrics((test_pred.predictions, test_pred.label_ids))
print(test_metrics)

{'accuracy': 0.860613810741688, 'macro_precision': 0.8013803857290268, 'macro_recall': 0.6468693996415771, 'macro_f1': 0.6662121659771176, 'weighted_precision': 0.8608761106525952, 'weighted_recall': 0.860613810741688, 'weighted_f1': 0.8530174658666402}


In [ ]:
report_path = os.path.join(model_save_path, "asc_phase2_benchmark.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("ASC Phase 2 Benchmark\n")
    f.write(f"Model: {model_name}\n")
    for k, v in test_metrics.items():
        f.write(f"{k}: {v:.6f}\n")

metrics_json = os.path.join(model_save_path, "asc_phase2_benchmark.json")
with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": model_name,
            **{k: float(v) for k, v in test_metrics.items()},
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved report: {report_path}")
print(f"Saved json   : {metrics_json}")

Saved report: /content/drive/MyDrive/data_train/asc/asc_model/asc_phase2_benchmark.txt
Saved json   : /content/drive/MyDrive/data_train/asc/asc_model/asc_phase2_benchmark.json
